In [ ]:
from easyAI import TwoPlayerGame, Human_Player, AI_Player, Negamax

class GameOfBones( TwoPlayerGame ):
    """ In turn, the players remove one, two or three bones from a
    pile of bones. The player who removes the last bone loses. """

    def __init__(self, players=None):
        self.players = players
        self.pile = 20 # start with 20 bones in the pile
        self.current_player = 1 # player 1 starts

    def possible_moves(self): return ['1','2','3']
    def make_move(self,move): self.pile -= int(move) # remove bones.
    def win(self): return self.pile<=0 # opponent took the last bone ?
    def is_over(self): return self.win() # Game stops when someone wins.
    def show(self): print ("%d bones left in the pile" % self.pile)
    def scoring(self): return 100 if game.win() else 0 # For the AI

# Start a match (and store the history of moves when it ends)
ai = Negamax(13) # The AI will think 13 moves in advance
game = GameOfBones( [ Human_Player(), AI_Player(ai) ] )
history = game.play()

20 bones left in the pile

Move #1: player 1 plays 3 :
17 bones left in the pile

Move #2: player 2 plays 1 :
16 bones left in the pile

Move #3: player 1 plays 2 :
14 bones left in the pile

Move #4: player 2 plays 1 :
13 bones left in the pile

Move #5: player 1 plays 3 :
10 bones left in the pile

Move #6: player 2 plays 1 :
9 bones left in the pile

Move #7: player 1 plays 2 :
7 bones left in the pile

Move #8: player 2 plays 1 :
6 bones left in the pile

Move #9: player 1 plays 2 :
4 bones left in the pile

Move #10: player 2 plays 1 :
3 bones left in the pile

Move #11: player 1 plays 2 :
1 bones left in the pile

Move #12: player 2 plays 1 :
0 bones left in the pile


In [1]:
try:
    import numpy as np
except ImportError:
    print("Sorry, this example requires Numpy installed !")
    raise

from easyAI import TwoPlayerGame, AI_Player, Negamax


class ConnectFour(TwoPlayerGame):
    """
    Connect Four with optional move noise.

    If noise_prob > 0, the chosen column can be randomly shifted
    by one to the left/right before placing the token.
    """

    def __init__(self, players, board=None, noise_prob=0.0, seed=None):
        self.players = players
        self.board = board if (board is not None) else np.zeros((6, 7), dtype=int)
        self.current_player = 1
        self.noise_prob = float(noise_prob)
        self.rng = np.random.default_rng(seed)

    def possible_moves(self):
        return [i for i in range(7) if self.board[:, i].min() == 0]

    def randomly_change_col(self, column):
        if self.noise_prob <= 0.0:
            return int(column)

        if self.rng.random() >= self.noise_prob:
            return int(column)

        candidates = []
        if column > 0:
            candidates.append(column - 1)
        if column < 6:
            candidates.append(column + 1)

        if not candidates:
            return int(column)

        return int(self.rng.choice(candidates))

    def make_move(self, column):
        base_column = int(column)
        column = self.randomly_change_col(base_column)

        legal = self.possible_moves()
        if column not in legal:
            column = base_column if base_column in legal else legal[0]

        line = int(np.where(self.board[:, column] == 0)[0][0])
        self.board[line, column] = self.current_player

    def show(self):
        print(
            "\n"
            + "\n".join(
                ["0 1 2 3 4 5 6", 13 * "-"]
                + [
                    " ".join([".", "O", "X"][self.board[5 - j][i]] for i in range(7))
                    for j in range(6)
                ]
            )
        )

    def lose(self):
        return find_four(self.board, self.opponent_index)

    def is_over(self):
        return (self.board.min() > 0) or self.lose()

    def scoring(self):
        return -100 if self.lose() else 0


def find_four(board, player_id):
    """Returns True iff given player has connected 4 or more."""
    for pos, direction in POS_DIR:
        streak = 0
        while (0 <= pos[0] <= 5) and (0 <= pos[1] <= 6):
            if board[pos[0], pos[1]] == player_id:
                streak += 1
                if streak == 4:
                    return True
            else:
                streak = 0
            pos = pos + direction
    return False


POS_DIR = np.array(
    [[[i, 0], [0, 1]] for i in range(6)]
    + [[[0, i], [1, 0]] for i in range(7)]
    + [[[i, 0], [1, 1]] for i in range(1, 3)]
    + [[[0, i], [1, 1]] for i in range(4)]
    + [[[i, 6], [1, -1]] for i in range(1, 3)]
    + [[[0, i], [1, -1]] for i in range(3, 7)]
)


def play_ai_game(depth_a, depth_b, starter_agent="A", noise_prob=0.0, seed=None, verbose=False):
    """
    Plays one game between two Negamax agents.

    Agent A uses depth_a, Agent B uses depth_b.
    starter_agent: "A" or "B".
    """
    player_a = AI_Player(Negamax(depth_a))
    player_b = AI_Player(Negamax(depth_b))

    if starter_agent == "A":
        players = [player_a, player_b]
        slot_to_agent = {1: "A", 2: "B"}
    else:
        players = [player_b, player_a]
        slot_to_agent = {1: "B", 2: "A"}

    game = ConnectFour(players, noise_prob=noise_prob, seed=seed)
    history = game.play(verbose=verbose)

    winner_slot = game.opponent_index if game.lose() else None
    winner_agent = slot_to_agent[winner_slot] if winner_slot is not None else None

    return {
        "winner_agent": winner_agent,
        "winner_slot": winner_slot,
        "starter_agent": starter_agent,
        "moves": max(0, len(history) - 1),
    }


def benchmark_negamax(
    depth_pairs=((2, 7), (1, 4)),
    games_per_pair=40,
    noise_levels=(0.0, 0.2),
    base_seed=1234,
):
    """
    Runs A-vs-B Negamax benchmarks for multiple depths and game variants.

    - Alternates starting player every game.
    - Compares deterministic (noise=0.0) and probabilistic variants.
    """
    rng = np.random.default_rng(base_seed)
    results = []
    game_count = 0
    max_games = len(depth_pairs) * len(noise_levels) * games_per_pair

    for noise_prob in noise_levels:
        variant = "deterministic" if noise_prob == 0.0 else f"probabilistic(p={noise_prob:.2f})"

        for depth_a, depth_b in depth_pairs:
            stats = {
                "variant": variant,
                "depth_a": depth_a,
                "depth_b": depth_b,
                "games": games_per_pair,
                "a_wins": 0,
                "b_wins": 0,
                "draws": 0,
                "slot1_wins": 0,
                "slot2_wins": 0,
                "a_wins_as_starter": 0,
                "a_wins_as_second": 0,
                "b_wins_as_starter": 0,
                "b_wins_as_second": 0,
                "total_moves": 0,
            }

            for game_idx in range(games_per_pair):
                game_count += 1
                starter_agent = "A" if game_idx % 2 == 0 else "B"
                game_seed = int(rng.integers(0, 2**32 - 1))

                outcome = play_ai_game(
                    depth_a=depth_a,
                    depth_b=depth_b,
                    starter_agent=starter_agent,
                    noise_prob=noise_prob,
                    seed=game_seed,
                    verbose=False,
                )

                stats["total_moves"] += outcome["moves"]

                if outcome["winner_slot"] == 1:
                    stats["slot1_wins"] += 1
                elif outcome["winner_slot"] == 2:
                    stats["slot2_wins"] += 1

                if outcome["winner_agent"] == "A":
                    stats["a_wins"] += 1
                    if outcome["starter_agent"] == "A":
                        stats["a_wins_as_starter"] += 1
                    else:
                        stats["a_wins_as_second"] += 1
                elif outcome["winner_agent"] == "B":
                    stats["b_wins"] += 1
                    if outcome["starter_agent"] == "B":
                        stats["b_wins_as_starter"] += 1
                    else:
                        stats["b_wins_as_second"] += 1
                else:
                    stats["draws"] += 1

                print(f"Played {game_count}/{max_games}")

            g = stats["games"]
            stats["a_win_rate"] = stats["a_wins"] / g
            stats["b_win_rate"] = stats["b_wins"] / g
            stats["draw_rate"] = stats["draws"] / g
            stats["avg_moves"] = stats["total_moves"] / g


            results.append(stats)

    return results


def print_benchmark(results):
    header = (
        f"{'variant':24} {'depth(A:B)':10} {'A_wins':>7} {'B_wins':>7} "
        f"{'draws':>7} {'A_rate':>8} {'B_rate':>8} {'draw_rate':>10} {'avg_moves':>10}"
    )
    print(header)
    print("-" * len(header))

    for r in results:
        print(
            f"{r['variant']:24} {r['depth_a']}:{r['depth_b']:<7} "
            f"{r['a_wins']:7d} {r['b_wins']:7d} {r['draws']:7d} "
            f"{r['a_win_rate']:8.2%} {r['b_win_rate']:8.2%} {r['draw_rate']:10.2%} {r['avg_moves']:10.2f}"
        )


results = benchmark_negamax(
    depth_pairs=[(3, 5), (4, 6)],
    games_per_pair=50,
    noise_levels=[0.0, 0.2],
    base_seed=42,
)
print_benchmark(results)
results




Played 1/200
Played 2/200
Played 3/200
Played 4/200
Played 5/200
Played 6/200
Played 7/200
Played 8/200
Played 9/200
Played 10/200
Played 11/200
Played 12/200
Played 13/200
Played 14/200
Played 15/200
Played 16/200
Played 17/200
Played 18/200
Played 19/200
Played 20/200
Played 21/200
Played 22/200
Played 23/200
Played 24/200
Played 25/200
Played 26/200
Played 27/200
Played 28/200
Played 29/200
Played 30/200
Played 31/200
Played 32/200
Played 33/200
Played 34/200
Played 35/200
Played 36/200
Played 37/200
Played 38/200
Played 39/200
Played 40/200
Played 41/200
Played 42/200
Played 43/200
Played 44/200
Played 45/200
Played 46/200
Played 47/200
Played 48/200
Played 49/200
Played 50/200
Played 51/200
Played 52/200
Played 53/200
Played 54/200
Played 55/200
Played 56/200
Played 57/200
Played 58/200
Played 59/200
Played 60/200
Played 61/200
Played 62/200
Played 63/200
Played 64/200
Played 65/200
Played 66/200
Played 67/200
Played 68/200
Played 69/200
Played 70/200
Played 71/200
Played 72/200
P

[{'variant': 'deterministic',
  'depth_a': 3,
  'depth_b': 5,
  'games': 50,
  'a_wins': 25,
  'b_wins': 25,
  'draws': 0,
  'slot1_wins': 0,
  'slot2_wins': 50,
  'a_wins_as_starter': 0,
  'a_wins_as_second': 25,
  'b_wins_as_starter': 0,
  'b_wins_as_second': 25,
  'total_moves': 1900,
  'a_win_rate': 0.5,
  'b_win_rate': 0.5,
  'draw_rate': 0.0,
  'avg_moves': 38.0},
 {'variant': 'deterministic',
  'depth_a': 4,
  'depth_b': 6,
  'games': 50,
  'a_wins': 25,
  'b_wins': 25,
  'draws': 0,
  'slot1_wins': 0,
  'slot2_wins': 50,
  'a_wins_as_starter': 0,
  'a_wins_as_second': 25,
  'b_wins_as_starter': 0,
  'b_wins_as_second': 25,
  'total_moves': 1900,
  'a_win_rate': 0.5,
  'b_win_rate': 0.5,
  'draw_rate': 0.0,
  'avg_moves': 38.0},
 {'variant': 'probabilistic(p=0.20)',
  'depth_a': 3,
  'depth_b': 5,
  'games': 50,
  'a_wins': 9,
  'b_wins': 33,
  'draws': 8,
  'slot1_wins': 18,
  'slot2_wins': 24,
  'a_wins_as_starter': 2,
  'a_wins_as_second': 7,
  'b_wins_as_starter': 16,
  'b_w

### Wyniki
Dwóch graczy AI z algorytmem Negamax
przeciwko sobie wielokrotnie, zmieniając gracza rozpoczynającego.

| variant               | depth(A:B) | A_wins | B_wins | draws | A_rate | B_rate | draw_rate | avg_moves |
|-----------------------|------------|--------|--------|-------|--------|--------|-----------|-----------|
| deterministic         | 2:7        | 25     | 25     | 0     | 50.00% | 50.00% | 0.00%     | 38.00     |
| deterministic         | 1:4        | 25     | 25     | 0     | 50.00% | 50.00% | 0.00%     | 38.00     |
| probabilistic(p=0.20) | 2:7        | 9      | 33     | 8     | 18.00% | 66.00% | 16.00%    | 30.28     |
| probabilistic(p=0.20) | 1:4        | 11     | 24     | 15    | 22.00% | 48.00% | 30.00%    | 35.82     |

